# 3장 체인을 구현하는 기초 문법, LCEL
#### 필요 패키지 설치
- 아래 코드 셀에 명령어를 입력하여 필요한 패키지를 설치합니다.

- 코랩 환경에서는 코드 셀에 `!` 기호를 사용하여 `pip`와 관련된 명령어를 수행하도록 할 수 있습니다.

In [ ]:
# 아래에 패키지 설치 명령어 작성하기


In [ ]:
# 아래에 패키지 버전 확인을 위한 명령어 작성하기
# pip show 패키지 이름으로 확인 할 수 있음


#### 오픈AI API 키를 OS 환경 변수에 입력

- 환경 변수에 API 키를 입력해 둠으로써, 앞으로 이 파일에서는 API 키를 다시 입력할 필요가 없습니다.

In [ ]:
# os 모듈 불러오기

# 자신의 오픈 AI API Key 입력
os.environ["OPENAI_API_KEY"] = "오픈AI API Key"

---

## 3.1 LCEL 기초 문법 이해
### 3.1.1 LCEL 체인의 구축 및 실행

#### 프롬프트 정의
- `from_temaplte` 메서드에 프롬프트 정의하기



#### LLM 모델 객체 생성
- 오픈 AI의 LLM 모델 불러오기
- API Key는 위에서 환경 변수에 설정하였으므로 별도 과정 필요 없음

#### 출력 파서 정의
- 출력 파서는 LLM 모델이 생성한 응답을 적절한 형태로 변환합니다.

#### 구성 요소 연결
- 파이프 `|` 연산자로 구성 요소를 연결합니다.

#### 최종 출력 생성

#### strem 메서드
- 챗GPT처럼 답변이 실시간으로 생성되는 듯 자연스러운 연출을 구현하고 싶다면, stream 메서드를 사용할 수 있습니다.

#### 프롬프트에 다수의 변수 정의하기
- {주제}에 대해 {난이도} 수준으로 설명해 달라는 프롬프트를 작성하기
- 각 변수 명은 `topic`과 `level`로 설정

In [ ]:
# 새로운 프롬프트 정의

# 새로운 체인 생성

# 체인 실행

---

## 3.2 LCEL 구축 시 주의 사항
### 3.2.1 다중 입력 변수 처리 및 KeyError
- 앞서 두 개의 입력 변수 `topic`과 `level`을 프롬프트에 정의한 경우, `invoke` 메서드를 호출할 때 하나라도 빠트리면 `KeyError`가 발생합니다.

In [ ]:
# 잘못된 호출 (level 변수 누락 -> KeyError 발생)
response_error = chain2.invoke({"topic": "양자역학"}) # KeyError 발생!

### 3.2.2 체인 구성 및 ValueError

- 체인은 반드시 프롬프트(prompt) -> 모델(model) -> 출력 파서(output_parser) 순서로 연결되어야 합니다.

In [ ]:
# 잘못된 체인 구성
chain2 = model | prompt2 | output_parser
chain2.invoke({"topic": "인공지능", "level" : "초등학생"}) # ValueError 발생!

---

## 3.3 결합된 체인 구축
#### 첫 번째 체인 생성
- 아직 invoke 메서드를 호출한 적은 없으므로, 이 딕셔너리의 값에는 chain1 객체만 할당
- 즉, 실제로 model을 호출한 적은 없는 것입니다.

In [ ]:
# 첫 번째 체인의 프롬프트

# 첫 번째 체인생성

# 새로운 딕셔너리 생성


#### 두 번째 체인 생성
- 이 프롬프트는 새롭게 만든 딕셔너리(chain1_result_dict)의 키(chain1_result)를 입력 변수로 가져야 합니다.

In [ ]:
# 두 번째 체인의 프롬프트

# combined_chain 생성


#### 결합된 체인 실행
- 첫 번째 체인의 프롬프트에 정의된 입력 변수 `topic`과 `level`을 딕셔너리 형태로 전달합니다.

---

## 3.4 프롬프트
- 프롬프트는 'prompt'의 의미(무언가를 지시하는 문장) 그대로 AI 모델에게 내리는 지시 또는 명령을 의미합니다.

### 3.4.1 PromptTemplate.from_template()
- 가장 단순하며 다양한 상황에서 융통성 있게 활용 가능한 방법

### 3.4.2 PromptTemplate
- from_template 메서드 없이 PromptTemplate를 직접 사용하는 방법

#### input_variables와 partial_variables 확인
- 주의할 점은 partial_variables에 사전 정의된 변수는 partial_variables 속성에서 별도로 확인하여야 합니다.

In [ ]:
print(prompt.input_variables)   # ['level']
print(prompt.partial_variables)    # {'topic': '양자역학'}

#### partial_variables 1
- 특정 변수의 값을 사전에 설정하기 위해 사용

In [ ]:

chain.invoke({"n": "3", "m":"5"})  # 답변 예시 : '2033입니다. 계산: 2025 + 3 = 2028, 2028 + 5 = 2033.'

#### partial_variables 2
- 고정된 값뿐만 아니라 함수도 연결할 수 있습니다.

In [ ]:
from datetime import datetime

# 올해 연도를 문자열로 반환하는 함수
def get_current_year():
    return str(datetime.now().year)

prompt = PromptTemplate(
    template= "{year} + {n} + {m}는 무엇인가요?",
    partial_variables={"year": get_current_year},
)
chain = prompt | model | output_parser
chain.invoke({"n": "3", "m":"5"})  # 답변 예시 : '2033입니다. 계산: 2025 + 3 = 2028, 2028 + 5 = 2033.'


#### partial_variables 주의 사항
- partial_variables에 `n`을 10으로 미리 할당했더라도, `invoke` 메서드 호출 시 새로운 `n` 값을 전달하면 메서드 호출시 전달된 값으로 업데이트되어 프롬프트가 완성됩니다

In [ ]:
prompt = PromptTemplate(
    template= "{year} + {n} + {m}는 무엇인가요?",
    partial_variables={
        "year": "2025",
        "n": "10"
    },
)

chain = prompt | model | output_parser
chain.invoke({"n": "3", "m":"5"}) # 답변 예시 : 2024 + 3 + 5은 2032입니다.

### 3.4.3 ChatPromptTemplate
- 이전 대화 내역을 추가함으로써 과거 대화와 연관된 답변을 생성하게 할 수 있게 합니다.

---

## 3.5 모델
#### 다양한 모델 사용하기
  - langchain-openai는 위에서 설치하였으므로 건너뜀

  - 아래는 예시로 코드로 별도로 실행하지 않습니다.

In [ ]:
!pip install -U langchain-anthropic

In [ ]:
from langchain_openai import ChatOpenAI
# from langchain_anthropic import ChatAnthropic

# claude 모델을 위한 API Key를 발급 받지 않았으므로,
# 이번 실습에서는 사용하지 않겠습니다.
# model = ChatAnthropic(model="claude-3-sonnet")
model = ChatOpenAI(model="gpt-5-nano")

#### 오픈AI 모델 하이퍼파라미터

- model: 오픈AI에서 제공하는 여러 LLM 모델 중 특정 모델을 지정합니다.
-	temperature: 생성되는 텍스트의 다양성을 조절하는 매개변수입니다.
  - 낮은 값(예: 0.2)은 더 결정적이고 일관된 출력을 생성
  - 높은 값(예: 0.8)은 더 창의적이고 다양성이 높은 출력을 생성합니다.
  - 높은 온도는 모델이 더 다양한 단어와 구문을 선택하도록 유도하여 창의적인 결과를 도출할 수 있습니다.
- max_tokens: 모델이 생성할 수 있는 최대 토큰 수를 지정합니다.
- max_retries: 모델 호출이 실패할 경우 재시도 횟수를 지정합니다.


#### 실습
- 생성한 모델의 하이퍼파라미터를 조정하며 다양한 답변 받아보기

In [ ]:
# 아래에 LCEL 문법에 맞춰 답변을 생성해 봅시다.


---

## 3.6 출력 파서
#### 출력 파서 없이 코드 실행하기
- 아래 코드를 실행하여 결과를 확인합니다.

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

prompt = PromptTemplate.from_template("{topic} 에 대해 쉽게 설명해 주세요.")
model = ChatOpenAI(
    model="gpt-5-nano",
    max_tokens=2048
    )

In [ ]:
chain = prompt | model
chain.invoke({"topic" : "인공지능"})

---

### CommaSeparatedOutputParser, JsonOutputParser 알아보기
#### 프롬프트 구성 예시

#### get_format_instructions 메서드와 활용 예시

---

### 3.6.1 CommaSeperatedListOutputParser
#### 콤마 구분자 출력 파서
1. 클래스 import

2. 프롬프트 불러오기

3. 프롬프트 완성

4. 모델 생성

5. 체인 연결과 실행

---

### 3.6.3 JsonOutputParser
1. 클래스 import

2. 프롬프트 불러오기

3. 프롬프트 완성

4. 체인 연결과 실행
- 실행할 때마다 JSON의 구조가 달라지는 문제가 있음
- 코드 작성 후, 여러번 실행하여 동일한 결과가 나오는지 확인합니다.

#### JSON 구조 직접 지정하기

- `chain.invoke({"name" : "블랙핑크"})`를 반복적으로 실행해 보며 결과를 확인해 봅시다.

In [ ]:
# 반복 실행 1
result = chain.invoke({"name" : "블랙핑크"})

print('출력 결과 : ', result)

In [ ]:
# 반복 실행 2
result = chain.invoke({"name" : "블랙핑크"})

print('출력 결과 : ', result)

---

### 참고. 랭체인 허브
- 사전 정의된 프롬프트 불러오기

In [ ]:
# Create a LANGSMITH_API_KEY in Settings > API Keys
from langsmith import Client


client = Client(api_key=LANGSMITH_API_KEY)
prompt = client.pull_prompt("hardkothari/prompt-maker", include_model=True)

prompt # 객체 확인

- 체인 구성

In [ ]:
chain = prompt | ChatOpenAI() | StrOutputParser()

task = """
반드시 한글로 작성되어야 합니다.
사용자의 질문을 읽고, 핵심 키워드를 파악해 전문 지식이 있는 사람의 질문으로 변경해 주세요.
더 체계적이고, 단계적인 질문이 될 수 있도록 변경하세요.
사용자의 질문에서 벗어나서는 안됩니다.
"""
lazy_prompt = "강아지 눈이 좀 이상해요"

improved_prompt = chain.invoke({"task" : task, "lazy_prompt" : lazy_prompt})

print(improved_prompt) # 개선된 프롬프트 확인

- 다른 질문으로 테스트하기

In [ ]:
chain = prompt | ChatOpenAI() | StrOutputParser()

task = """
반드시 한글로 작성되어야 합니다.
사용자의 질문을 읽고, 핵심 키워드를 파악해 전문 지식이 있는 사람의 질문으로 변경해 주세요.
더 체계적이고, 단계적인 질문이 될 수 있도록 변경하세요.
사용자의 질문에서 벗어나서는 안됩니다.
"""
lazy_prompt = "여기에 다른 주제의 질문을 작성해 보세요."

improved_prompt = chain.invoke({"task" : task, "lazy_prompt" : lazy_prompt})

print(improved_prompt) # 개선된 프롬프트 확인